In [1]:
print("GRACE")

GRACE


In [5]:
from flask import Flask, request, render_template, redirect, url_for, jsonify
from flask_sqlalchemy import SQLAlchemy
from flask import Flask, request, render_template, redirect, url_for, send_file
import os
import pdfkit
import pdfkit
from PyPDF2 import PdfMerger
from datetime import datetime
from werkzeug.utils import secure_filename
from utils import generate_cover_pdf, generate_second_page_with_info, reference_images_to_pdf
from werkzeug.security import generate_password_hash, check_password_hash
from flask import send_from_directory
import os
from flask import flash
import os
import shutil

from sqlalchemy.orm import aliased
from PIL import Image, ImageDraw, ImageFont
import platform
import os

ModuleNotFoundError: No module named 'utils'

In [8]:
import fitz  # PyMuPDF


In [21]:



def fill_blanks_with_coordinates(form_id,fill_values):
    input_pdf = "risk_assestment_matrix_2.pdf"
    coordinates = [
    (109, 415),  # x=100, y=200
    (251, 415),  # x=200, y=200
    (352, 415),  # x=300, y=200
    ]

    output_pdf_path = f"risk_assestment_matrix_output{form_id}.pdf"
    """
    Fill blanks in a PDF by placing text at specified coordinates.

    :param input_pdf_path: Path to the input PDF.
    :param output_pdf_path: Path to save the modified PDF.
    :param fill_values: List of values to fill in the blanks.
    :param coordinates: List of tuples with coordinates (x, y) for each value.
    """
    # Open the PDF
    pdf_document = fitz.open(input_pdf)
    page = pdf_document[0]  # Assuming there is only one page

    # Iterate over each blank to fill
    for i, (x, y) in enumerate(coordinates):
        if i < len(fill_values):
            text = str(fill_values[i])
            page.insert_text((x, y), text, fontsize=15, color=(0, 0, 0))

    # Save the modified PDF
    pdf_document.save(output_pdf_path)
    pdf_document.close()


In [22]:
fill_blanks_with_coordinates(1, [1,2,3])

In [1]:
from PIL import Image, ImageDraw, ImageFont
import os
from datetime import datetime
import reportlab
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.lib.units import inch

from PIL import Image, ImageDraw, ImageFont
import os

import platform

In [11]:
def generate_cover_pdf(form_id, property_name):
    # Create download/property_cover_page directory if it doesn't exist
    cover_page_dir = os.path.join("downloads", "property_cover_page")
    os.makedirs(cover_page_dir, exist_ok=True)
    
    # Define PDF path in the new directory
    pdf_path = os.path.join(cover_page_dir, f"cover_{form_id}.pdf")
    
    # Create canvas
    c = canvas.Canvas(pdf_path, pagesize=A4)
    width, height = A4
    
    # Background color
    c.setFillColorRGB(1, 1, 1)  # White background
    c.rect(0, 0, width, height, fill=1, stroke=0)
    
    # Darker Red Color for Highlight
    dark_red_color = (0.9, 0, 0)  # Darker red RGB
    
    # Slimmer Highlighted Area for Property Name
    highlight_height = 0.5 * inch  # Reduced height for a slimmer highlighted area
    header_margin = 50  # Margin from the sides
    
    # Calculate the y-position of the highlighted area
    highlight_y_position = height - 1.8 * inch - highlight_height  # Adjusted y-position
    
    # Draw the highlighted rectangle
    c.setFillColorRGB(*dark_red_color)  # Use darker red color
    c.rect(header_margin, highlight_y_position, width - 2 * header_margin, highlight_height, fill=1, stroke=0)
    
    # Property Name Header Text (white)
    c.setFillColorRGB(1, 1, 1)  # White color for text
    c.setFont("Helvetica-Bold", 24)  # Larger font size
    
    # Adjust the vertical position of the text to center it within the highlighted area
    text_y_position = highlight_y_position + (highlight_height / 2) - 8  # Adjusted for vertical centering
    c.drawCentredString(width / 2, text_y_position, property_name)
    
    # Try to find and add cover image
    cover_image_folder = os.path.join("uploads", form_id, "cover_image")
    cover_images = [f for f in os.listdir(cover_image_folder) if f.startswith(f"building_cover_image_{form_id}")]
    
    if cover_images:
        cover_image_path = os.path.join(cover_image_folder, cover_images[0])
        
        # Open image to get dimensions
        from PIL import Image
        img = Image.open(cover_image_path)
        img_width, img_height = img.size
        
        # Adjust image size (increased from 1/3 to 1/2 of A4 width for a larger image)
        img_display_size = width / 2
        aspect_ratio = img_height / img_width
        scaled_height = img_display_size * aspect_ratio
        
        # Center the image
        x_centered = (width - img_display_size) / 2
        y_positioned = height / 2 - scaled_height / 2
        
        # Draw the image
        c.drawImage(cover_image_path, x_centered, y_positioned, width=img_display_size, height=scaled_height)
    
    # Property Details Below Image (dark black color)
    c.setFillColorRGB(0, 0, 0)  # Dark black color
    c.setFont("Helvetica-Bold", 16)
    c.drawCentredString(width / 2, height / 2 - scaled_height / 2 - 50, f"Property: {property_name}")
    
    # Form ID
    c.setFont("Helvetica", 12)
    c.drawCentredString(width / 2, height / 2 - scaled_height / 2 - 80, f"Assessment ID: {form_id}")
    
    # Date
    current_date = datetime.now().strftime("%d %B %Y")
    c.drawCentredString(width / 2, height / 2 - scaled_height / 2 - 110, f"Date: {current_date}")
    
    # Footer (dark black color)
    c.setFillColorRGB(0, 0, 0)  # Dark black color
    c.setFont("Helvetica", 10)
    c.drawCentredString(width / 2, 50, "© 2024 Amin Constructions. All Rights Reserved.")
    
    # Save the PDF
    c.save()
    return pdf_path

In [12]:
generate_cover_pdf("1", "Test New Heading Highlight")

'downloads/property_cover_page/cover_1.pdf'

In [36]:
import os
from reportlab.lib.pagesizes import A4 
from reportlab.lib.units import inch 
from reportlab.pdfgen import canvas 
from PIL import Image
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import Paragraph
from reportlab.lib.enums import TA_CENTER

def reference_images_to_pdf(form_id):
    # Path to the folder containing images
    image_folder = f"uploads/{form_id}"
    
    # Output PDF path
    output_pdf_path = f"downloads/reference_pictures_{form_id}.pdf"
    
    # Ensure downloads directory exists
    os.makedirs(os.path.dirname(output_pdf_path), exist_ok=True)
    
    # Create PDF canvas
    c = canvas.Canvas(output_pdf_path, pagesize=A4)
    width, height = A4
    
    # Margins
    margin_x = 0.75 * inch  # Symmetric left and right margins
    margin_y = 0.5 * inch  # Top and bottom margins
    
    # Horizontal and vertical spacing
    horizontal_padding = 0.5 * inch  # Space between columns
    vertical_padding = 0.3 * inch  # Space between rows
    
    # Calculate image size to fit two columns with symmetric margins
    img_size = (width - 2 * margin_x - horizontal_padding) / 2
    
    # White background for the entire page
    c.setFillColorRGB(1, 1, 1)
    c.rect(0, 0, width, height, fill=1, stroke=0)
    
    # Header for Reference Images
    header_text = "Reference Images"
    header_font_size = 20
    header_font = "Helvetica-Bold"
    
    # Calculate header dimensions
    text_width = c.stringWidth(header_text, header_font, header_font_size)
    text_height = header_font_size  # Approximate height of the text
    header_padding = 0.2 * inch  # Padding around the text
    
    # Dark red background for the header
    c.setFillColorRGB(0.9, 0, 0)  # Dark red color
    c.rect(
        (width - text_width) / 2 - header_padding,  # X position (centered)
        height - margin_y - text_height - header_padding,  # Y position
        text_width + 2 * header_padding,  # Width of the background
        text_height + 2 * header_padding,  # Height of the background
        fill=1,
        stroke=0
    )
    
    # White text for the header
    c.setFillColorRGB(1, 1, 1)  # White color
    c.setFont(header_font, header_font_size)
    c.drawCentredString(width / 2, height - margin_y - text_height, header_text)
    
    # Get list of image files
    image_files = [f for f in os.listdir(image_folder)
                   if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp'))]
    
    # Caption style
    caption_style = ParagraphStyle(
        'CaptionStyle',
        fontName='Helvetica',
        fontSize=10,
        textColor='black',
        alignment=TA_CENTER,
        leading=12  # Line height
    )
    
    # Track vertical position - minimal gap, start very close to the header
    y_position = height - margin_y - 0.2 * inch - (text_height + 2 * header_padding)
    
    # X positions for two columns with symmetric margins
    x_positions = [
        margin_x, 
        margin_x + img_size + horizontal_padding
    ]
    current_column = 0
    
    # Add images to PDF
    for filename in image_files:
        # Full path to image
        image_path = os.path.join(image_folder, filename)
        
        # Open image to get dimensions
        img = Image.open(image_path)
        img_width, img_height = img.size
        
        # Calculate scaled image size maintaining aspect ratio
        aspect_ratio = img_height / img_width
        scaled_height = img_size * aspect_ratio
        
        # Draw image
        c.drawImage(image_path, x_positions[current_column], y_position - scaled_height,
                    width=img_size, height=scaled_height, preserveAspectRatio=True)
        
        # Add image name below the image
        image_name = os.path.splitext(filename)[0]  # Remove file extension
        
        # Prepare paragraph for caption
        para = Paragraph(image_name, caption_style)
        
        # Calculate paragraph height
        para_width = img_size
        para_height = para.wrap(para_width, 100)[1]
        
        # Position caption below the image with some padding
        caption_y = y_position - scaled_height - 15 - para_height
        
        # Draw the paragraph
        para.drawOn(c, x_positions[current_column], caption_y)
        
        # Move to next column/row
        current_column += 1
        if current_column > 1:
            current_column = 0
            y_position -= scaled_height + vertical_padding + para_height + 30  # Move to next row
    
    # Save PDF
    c.save()

    return output_pdf_path

In [37]:
reference_images_to_pdf(2)

'downloads/reference_pictures_2.pdf'